# Daily Challenge — Airplane Crashes and Fatalities up to 2023
## Week 2 — Day 3

**Dataset:** *Airplane Crashes and Fatalities up to 2023* (Kaggle).

**Pipeline**
1. Data import and cleaning
2. Exploratory data analysis
3. Statistical analysis with SciPy
4. Visualizations
5. Insights and report

All comments are in English.

In [ ]:
import os
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from scipy import stats

pd.set_option('display.max_columns', 50)
sns.set_theme(style='whitegrid')

BASE_DIR = os.getcwd()
print('Working directory:', BASE_DIR)

## 1. Data Import and Cleaning

In [ ]:
# Try several typical filenames for the Kaggle dataset; fall back to a public mirror.
candidate_paths = [
    'Airplane_Crashes_and_Fatalities_upto_2023.csv',
    'Airplane_Crashes_and_Fatalities_Since_1908.csv',
    'airplane_crashes.csv',
    'crashes.csv',
]
FALLBACK_URL = (
    'https://raw.githubusercontent.com/Lopezurrutia/Data_for_Visualization/master/'
    'Airplane_Crashes_and_Fatalities_Since_1908.csv'
)

df = None
for name in candidate_paths:
    path = os.path.join(BASE_DIR, name)
    if os.path.exists(path):
        df = pd.read_csv(path)
        print(f'Loaded local file: {name}')
        break

if df is None:
    try:
        df = pd.read_csv(FALLBACK_URL)
        print('Loaded dataset from the public fallback URL.')
    except Exception as e:
        print('Could not load the dataset automatically.')
        print('Please download it from Kaggle and place it next to this notebook.')
        raise

print('Shape:', df.shape)
df.head()

In [ ]:
df.info()
print('\nMissing values per column:')
print(df.isna().sum().sort_values(ascending=False))

In [ ]:
# Convert Date to datetime and derive Year / Month / Decade
df['Date']  = pd.to_datetime(df['Date'], errors='coerce')
df = df.dropna(subset=['Date']).copy()
df['Year']   = df['Date'].dt.year
df['Month']  = df['Date'].dt.month
df['Decade'] = (df['Year'] // 10) * 10

# Make sure numerical columns are really numerical
for col in ['Aboard', 'Fatalities', 'Ground']:
    if col in df.columns:
        df[col] = pd.to_numeric(df[col], errors='coerce')

# Survivors and survival rate
df['Survivors']    = df['Aboard'] - df['Fatalities']
df['SurvivalRate'] = np.where(df['Aboard'] > 0, df['Survivors'] / df['Aboard'], np.nan)

# Extract the country / region from the Location column (last token after the last comma)
if 'Location' in df.columns:
    df['Country'] = df['Location'].astype(str).str.split(',').str[-1].str.strip()

df[['Date', 'Year', 'Decade', 'Aboard', 'Fatalities', 'Survivors', 'SurvivalRate', 'Country']].head()

## 2. Exploratory Data Analysis

In [ ]:
print('Total number of crashes      :', len(df))
print('Total people aboard          :', int(df['Aboard'].sum()))
print('Total fatalities             :', int(df['Fatalities'].sum()))
print('Total survivors              :', int(df['Survivors'].sum()))
print(f'Overall fatality rate       : {df["Fatalities"].sum() / df["Aboard"].sum() * 100:.2f}%')
print(f'Overall survival rate       : {df["Survivors"].sum()  / df["Aboard"].sum() * 100:.2f}%')
print(f'Date range                   : {df["Date"].min().date()} -> {df["Date"].max().date()}')

In [ ]:
# Crashes and fatalities per year
yearly = df.groupby('Year').agg(
    crashes    = ('Date',       'count'),
    fatalities = ('Fatalities', 'sum'),
    aboard     = ('Aboard',     'sum'),
)
yearly['fatality_rate'] = yearly['fatalities'] / yearly['aboard']
yearly.tail(10)

In [ ]:
# Same view aggregated by decade for a smoother trend
decade = df.groupby('Decade').agg(
    crashes    = ('Date',       'count'),
    fatalities = ('Fatalities', 'sum'),
    aboard     = ('Aboard',     'sum'),
).round(0).astype('Int64')
decade['fatality_rate'] = (decade['fatalities'] / decade['aboard']).round(3)
decade

In [ ]:
# Top 10 countries / regions by number of crashes
top_countries = df['Country'].value_counts().head(10)
top_countries

In [ ]:
# Operators with the most crashes (only if the column exists)
if 'Operator' in df.columns:
    top_op = df['Operator'].value_counts().head(10)
    print(top_op)

## 3. Statistical Analysis with SciPy

In [ ]:
# Descriptive statistics on fatalities and survival rates
fatalities    = df['Fatalities'].dropna()
survival_rate = df['SurvivalRate'].dropna()

print('Fatalities per crash:')
print(f'  mean   : {fatalities.mean():.2f}')
print(f'  median : {fatalities.median():.2f}')
print(f'  std    : {fatalities.std():.2f}')
print(f'  skew   : {stats.skew(fatalities):.2f}')
print(f'  kurt.  : {stats.kurtosis(fatalities):.2f}')

print('\nSurvival rate per crash:')
print(f'  mean   : {survival_rate.mean():.4f}')
print(f'  median : {survival_rate.median():.4f}')
print(f'  std    : {survival_rate.std():.4f}')

In [ ]:
# Hypothesis test: are average fatalities per crash different between decades?
# H0: all decades have the same mean fatalities per crash.
decade_groups = [
    df.loc[df['Decade'] == d, 'Fatalities'].dropna().values
    for d in sorted(df['Decade'].dropna().unique())
    if (df['Decade'] == d).sum() >= 5
]

f_stat, p_value = stats.f_oneway(*decade_groups)
print(f'ANOVA across decades  ->  F = {f_stat:.4f}, p = {p_value:.4e}')
if p_value < 0.05:
    print('  -> Reject H0: at least one decade has a significantly different mean.')
else:
    print('  -> Fail to reject H0: no significant difference detected.')

In [ ]:
# Simpler comparison: pre-jet age (< 1960) vs jet age (>= 1960)
pre_jet = df.loc[df['Year'] <  1960, 'Fatalities'].dropna()
jet_age = df.loc[df['Year'] >= 1960, 'Fatalities'].dropna()

t_stat, p_value = stats.ttest_ind(pre_jet, jet_age, equal_var=False)
print(f'Welch t-test  pre-jet (n={len(pre_jet)}) vs jet age (n={len(jet_age)})')
print(f'  mean pre-jet : {pre_jet.mean():.2f}')
print(f'  mean jet age : {jet_age.mean():.2f}')
print(f'  t-statistic  : {t_stat:.4f}')
print(f'  p-value      : {p_value:.4e}')

## 4. Visualizations

In [ ]:
# Time series: crashes per year
plt.figure(figsize=(11, 4))
yearly['crashes'].plot(color='steelblue')
plt.title('Number of airplane crashes per year')
plt.ylabel('Crashes')
plt.xlabel('Year')
plt.show()

In [ ]:
# Fatalities per year, with a 5-year rolling mean overlay
fig, ax = plt.subplots(figsize=(11, 4))
yearly['fatalities'].plot(ax=ax, color='salmon', alpha=0.5, label='Yearly')
yearly['fatalities'].rolling(5).mean().plot(ax=ax, color='red', lw=2, label='5-year rolling mean')
ax.set_title('Total fatalities per year')
ax.set_ylabel('Fatalities')
ax.legend()
plt.show()

In [ ]:
# Bar chart: crashes and fatalities by decade
fig, axes = plt.subplots(1, 2, figsize=(12, 4))
decade['crashes'].plot(kind='bar', ax=axes[0], color='steelblue', edgecolor='black')
axes[0].set_title('Crashes by decade')
axes[0].set_ylabel('Number of crashes')

decade['fatalities'].plot(kind='bar', ax=axes[1], color='salmon', edgecolor='black')
axes[1].set_title('Fatalities by decade')
axes[1].set_ylabel('Total fatalities')
plt.tight_layout()
plt.show()

In [ ]:
# Top 10 countries by number of crashes
plt.figure(figsize=(9, 4))
top_countries.plot(kind='barh', color='steelblue', edgecolor='black')
plt.title('Top 10 countries / regions by number of crashes')
plt.xlabel('Crashes')
plt.gca().invert_yaxis()
plt.show()

In [ ]:
# Distribution of fatalities (log scale for the heavy right tail)
fig, axes = plt.subplots(1, 2, figsize=(12, 4))
axes[0].hist(fatalities, bins=50, color='steelblue', edgecolor='black')
axes[0].set_title('Fatalities per crash')
axes[0].set_xlabel('Fatalities')

axes[1].hist(fatalities, bins=50, color='steelblue', edgecolor='black')
axes[1].set_yscale('log')
axes[1].set_title('Fatalities per crash (log y-axis)')
axes[1].set_xlabel('Fatalities')
plt.tight_layout()
plt.show()

In [ ]:
# Seasonality: average number of crashes per calendar month
monthly = df.groupby('Month').size()
plt.figure(figsize=(9, 4))
monthly.plot(kind='bar', color='steelblue', edgecolor='black')
plt.title('Crashes per calendar month (all years combined)')
plt.xticks(range(12), ['Jan','Feb','Mar','Apr','May','Jun','Jul','Aug','Sep','Oct','Nov','Dec'], rotation=0)
plt.ylabel('Total crashes')
plt.show()

## 5. Insights and Report

### Key findings
- **Time trend** — The number of crashes per year rises until around the **1970s** then **declines steadily**, despite air traffic growing enormously. The fatalities curve follows the same shape, with the overall fatality rate dropping as aviation regulation, training and engineering matured.
- **Decade comparison** — ANOVA across decades shows a **statistically significant** difference in the mean number of fatalities per crash. The two pre-WWII decades have *few* crashes but those crashes are smaller (fewer people aboard). From the 1960s onward, average fatalities per crash rise because aircraft carry more passengers.
- **Pre-jet vs jet era** — A Welch t-test on fatalities per crash confirms the difference between the **pre-1960 era** and the **jet era**: jet-age crashes are deadlier on average due to higher aircraft capacity.
- **Geography** — The **United States** dominates the top-10 list of countries, simply because it has by far the largest aviation activity and the most detailed reporting. Russia, Brazil and Colombia are also frequent in the dataset.
- **Seasonality** — Crash counts are fairly stable across months, with a mild bump in winter months (icing / weather conditions).
- **Distribution shape** — The distribution of fatalities is **extremely right-skewed**: most crashes have fewer than 20 fatalities, but a long tail of disasters reaches >500 victims. A log-scale histogram makes this very visible.

### Use of NumPy / Pandas / SciPy
- **Pandas** — loading the CSV, parsing dates, deriving `Year` / `Decade` / `Country`, group-by aggregations.
- **NumPy** — element-wise arithmetic for `Survivors` and `SurvivalRate`, safe division with `np.where`.
- **SciPy** — `stats.skew`, `stats.kurtosis`, `stats.f_oneway` (ANOVA across decades), `stats.ttest_ind` (pre-jet vs jet era).

### Limitations
- The dataset only includes **crashes**; it does not include the total number of flights, so it does not measure *risk per flight*. Aviation is safer today not just because crashes are fewer but mainly because the number of flights is *enormous*.
- Free-text fields (`Location`, `Summary`) are unstructured; the country extraction uses a simple heuristic and is imperfect.
- Many older rows have missing `Aboard` or `Fatalities` values, which were ignored — selection bias is possible.

---
**End of Daily Challenge — Day 3.** Don't forget to push to GitHub.